# Imports

In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Preprocessing

In [2]:
df = pd.read_csv('no_sales_df.csv')
df.head()

,Unnamed: 0.1,Unnamed: 0,Order Date,Ship Date,Ship Mode,Segment,City,State,Country,Market,...,Discount,Profit,Shipping Cost,Order Priority,Price,Avg_Sales_Category,Avg_Sales_Country,Avg_Sales_Market,Avg_Sales_Region,Discount_value
0,0,234,2012-03-30,2012-04-01,First Class,Consumer,Cairo,Al Qahirah,Egypt,Africa,...,0.0,140.1600,399.96,Critical,637.35,467.858939,172.770678,170.868370,170.868370,0.00
1,1,238,2014-12-23,2014-12-26,First Class,Consumer,Detroit,Michigan,United States,US,...,0.0,412.5394,397.52,High,226.67,416.248905,229.858001,229.858001,253.872674,0.00
2,2,239,2014-07-04,2014-07-04,Same Day,Home Office,Seattle,Washington,United States,US,...,0.2,209.5800,396.92,High,399.20,467.858939,229.858001,229.858001,226.493233,479.04
3,3,240,2014-11-21,2014-11-23,First Class,Consumer,New York City,New York,United States,US,...,0.0,327.5922,394.57,Critical,419.99,467.858939,229.858001,229.858001,238.336110,0.00
4,4,241,2013-01-23,2013-01-27,Standard Class,Consumer,Baku,Baki,Azerbaijan,EMEA,...,0.0,946.6800,393.62,High,514.50,416.248905,194.190000,160.302508,160.302508,0.00


In [3]:
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

In [7]:
X = df.drop(['Price', 'Unnamed: 0.1', 'Unnamed: 0'], axis=1)
y = df['Price']

In [8]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

# GridSearch

In [10]:
param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5],
    'iterations': [500, 1000, 1500]
}

model = CatBoostRegressor(
    loss_function='RMSE',
    random_state=42,
    cat_features=categorical_features,
    verbose=0
)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_val, y_val)
best_params = grid_search.best_params_

/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.chec

# Training

In [11]:
best_model = CatBoostRegressor(
    **best_params,
    loss_function='RMSE',
    random_state=42,
    cat_features=categorical_features,
    verbose=100
)

best_model.fit(
    X_train,
    y_train,
    early_stopping_rounds=50,
    use_best_model=True,
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

You should provide test set for use best model. use_best_model parameter has been switched to false value.


0:	learn: 90.6554057	total: 17.4ms	remaining: 26.1s
100:	learn: 32.0665070	total: 951ms	remaining: 13.2s
200:	learn: 28.2873199	total: 1.85s	remaining: 11.9s
300:	learn: 26.5271404	total: 2.76s	remaining: 11s
400:	learn: 25.5681118	total: 3.7s	remaining: 10.1s
500:	learn: 24.8410675	total: 4.65s	remaining: 9.27s
600:	learn: 24.2875618	total: 5.58s	remaining: 8.35s
700:	learn: 23.8039869	total: 6.51s	remaining: 7.43s
800:	learn: 23.4269676	total: 7.44s	remaining: 6.49s
900:	learn: 23.0846441	total: 8.37s	remaining: 5.56s
1000:	learn: 22.7209479	total: 9.3s	remaining: 4.63s
1100:	learn: 22.4483260	total: 10.3s	remaining: 3.71s
1200:	learn: 22.1850777	total: 11.2s	remaining: 2.79s
1300:	learn: 21.8893656	total: 12.2s	remaining: 1.86s
1400:	learn: 21.6690335	total: 13.2s	remaining: 930ms
1499:	learn: 21.4390043	total: 14.1s	remaining: 0us


In [12]:
y_test_pred = best_model.predict(X_test)

mae_ridge = mean_absolute_error(y_test, y_test_pred)
mse_ridge = mean_squared_error(y_test, y_test_pred)
rmse_ridge = mean_squared_error(y_test, y_test_pred, squared=False)
r2_ridge = r2_score(y_test, y_test_pred)

print('Metrics:')
print(f"MAE: {mae_ridge}")
print(f"MSE: {mse_ridge}")
print(f"RMSE: {rmse_ridge}")
print(f"R² Score: {r2_ridge}")

Metrics:
MAE: 12.322073556760921
MSE: 706.620354308027
RMSE: 26.582331619104203
R² Score: 0.9259116686674012


/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
